# Ride-Pooling Survey Regression
## 主模型精简版 v2（Q9 修正版，Google Colab）

这个版本已经修正：

- **Commute distance 固定使用 Q9**
- 增加变量映射重复检查，防止 Q9/Q34 再次误配
- 保留：Ordinal Logistic 主模型、M1/M2/M3、Odds Ratio、forest plot
- 不包含 binary logit / cross-validation / operational simulation

请从上到下一小块一小块运行。


## 1. 安装读取 SPSS 所需的包


In [ ]:
# Colab 默认通常没有 pyreadstat
# 用它读取 .sav 文件

!pip -q install pyreadstat


## 2. 导入 Python 包


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadstat

from scipy.stats import chi2
from statsmodels.miscmodels.ordinal_model import OrderedModel

print("所有包导入成功。")


## 3. 上传 SPSS 文件


In [ ]:
# 请上传：
# survey_encoded.sav

from google.colab import files

uploaded = files.upload()
sav_path = next(iter(uploaded.keys()))

print("已上传：", sav_path)


## 4. 读取 SPSS 数据


In [ ]:
# apply_value_formats=False：
# 保留 SPSS 原始数值编码，例如 1、2、3、4、5
# 后面需要按照问卷原始编码重新处理

df, meta = pyreadstat.read_sav(
    sav_path,
    apply_value_formats=False
)

print("样本数 N =", len(df))
print("变量数 =", df.shape[1])

df.head()


## 5. 查看 SPSS 变量名和中文标签


In [ ]:
# 建立：
# SPSS变量名 -> 中文题目标签

labels = {}

for col in meta.column_names:
    label = meta.column_names_to_labels.get(col)
    labels[col] = "" if label is None else str(label)

label_table = pd.DataFrame({
    "SPSS变量名": list(labels.keys()),
    "变量标签": list(labels.values())
})

label_table.head(30)


## 6. 自动找到本研究需要的问卷题目


In [ ]:
def find_col(keyword, fallback_question_number=None):
    """
    根据中文题目关键词寻找 SPSS 对应变量。
    如果找不到，再尝试根据题号匹配。
    """

    candidates = []

    for col in df.columns:
        text = f"{col} {labels.get(col, '')}"

        if keyword in text:
            candidates.append(col)

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        # 如果找到多个候选变量，
        # 优先选择中文标签更完整的那一个
        candidates = sorted(
            candidates,
            key=lambda c: len(labels.get(c, "")),
            reverse=True
        )
        return candidates[0]

    if fallback_question_number is not None:
        q = str(fallback_question_number)

        for col in df.columns:
            text = f"{col} {labels.get(col, '')}".lower()

            if f"q{q}" in text or f"{q}、" in text or f"{q}." in text:
                return col

    raise KeyError(f"找不到变量：{keyword}")


## 7. 指定 X 和 Y 对应的问卷题目


In [ ]:
COL = {
    # 基础特征
    "age": find_col("请问您的年龄", 6),
    "income": find_col("月收入是多少", 7),
    "commute": "Q9",  # 固定为 Q9，避免误匹配到 Q34
    "vehicle": find_col("车辆保有量", 10),

    # Behavioural variables
    "discount": find_col("拼车的折扣达到多少", 16),
    "try_pool": find_col("是否愿意尝试拼车出行", 24),
    "stranger": find_col("是否愿意与陌生人拼车", 29),
    "pay_willing": find_col("是否愿意为拼车服务支付一定的费用", 33),
    "wtp": find_col("愿意为拼车服务支付的最高价格", 34),
}

for key, col in COL.items():
    print(f"{key:12s} -> {col} | {labels.get(col, '')}")


# -----------------------------
# 变量映射安全检查
# -----------------------------
# 如果两个研究变量意外指向同一个 SPSS 列，立即报错，
# 避免后面出现完全共线性（例如之前 commute 和 WTP 都误指向 Q34）。
mapped_cols = list(COL.values())

if len(mapped_cols) != len(set(mapped_cols)):
    duplicates = [
        col for col in set(mapped_cols)
        if mapped_cols.count(col) > 1
    ]
    raise ValueError(
        f"发现重复变量映射：{duplicates}。请检查 COL 设置。"
    )

print("\n变量映射检查通过：没有重复使用同一 SPSS 变量。")


## 8. 提取基础 X


In [ ]:
def to_num(col):
    # 转为数值；
    # 不能转换的值自动设为 NaN
    return pd.to_numeric(
        df[col],
        errors="coerce"
    )


dat = pd.DataFrame({
    "age_level": to_num(COL["age"]),
    "income_level": to_num(COL["income"]),
    "commute_level": to_num(COL["commute"]),
    "vehicle_level": to_num(COL["vehicle"]),
})

dat.head()


## 9. 提取 Behavioural X 和 Y


In [ ]:
# Q16：最低可接受折扣
dat["discount_code"] = to_num(COL["discount"])

# Q24：拼车意愿（Y）
dat["try_pool_code"] = to_num(COL["try_pool"])

# Q29：与陌生人拼车意愿
dat["stranger_code"] = to_num(COL["stranger"])

# Q33：是否愿意付费
dat["pay_willing_code"] = to_num(COL["pay_willing"])

# Q34：最高愿付价格 WTP
dat["wtp_code_raw"] = to_num(COL["wtp"])

dat.head()


## 10. 对 Y：Q24 拼车意愿重新编码


In [ ]:
# 原始 Q24：
# 1 = 非常愿意
# 2 = 愿意
# 3 = 一般
# 4 = 不太愿意
# 5 = 非常不愿意
#
# 为了让模型结果更直观：
# 数值越大 = 越愿意拼车
#
# 所以反向编码：
# 1 -> 5
# 2 -> 4
# 3 -> 3
# 4 -> 2
# 5 -> 1

dat["try_willing"] = 6 - dat["try_pool_code"]

dat[
    ["try_pool_code", "try_willing"]
].head(10)


## 11. 对 Q29：与陌生人拼车意愿重新编码


In [ ]:
# Q29 同样反向编码：
# 数值越大 = 越愿意和陌生人拼车

dat["share_willing"] = 6 - dat["stranger_code"]

dat[
    ["stranger_code", "share_willing"]
].head(10)


## 12. 处理 Q34：WTP


In [ ]:
# Q34 是区间，不是精确连续 WTP：
#
# 1 = <5 RMB
# 2 = 5-10 RMB
# 3 = 10-20 RMB
# 4 = >20 RMB
#
# 如果 Q33 回答“不愿意支付”，
# Q34 通常是空值。
#
# 这里把这部分人定义为 WTP = 0 类，
# 而不是直接删除。

dat["wtp_level"] = dat["wtp_code_raw"].copy()

dat.loc[
    (dat["pay_willing_code"] == 2)
    & (dat["wtp_level"].isna()),
    "wtp_level"
] = 0

dat[
    ["pay_willing_code", "wtp_code_raw", "wtp_level"]
].head(20)


## 13. 形成最终分析数据


In [ ]:
model_vars = [
    "age_level",
    "income_level",
    "commute_level",
    "vehicle_level",

    "discount_code",
    "wtp_level",
    "share_willing",

    "try_willing"
]

analysis = dat[
    model_vars
].dropna().copy()

print("最终用于主模型的样本数 =", len(analysis))

analysis.head()


# 检查是否因为缺失值损失了样本
if len(analysis) < len(dat):
    print(f"注意：有 {len(dat) - len(analysis)} 个样本因模型变量缺失被删除。")
else:
    print("很好：没有因为模型变量缺失而删除样本。")


## 14. 查看核心变量分布


In [ ]:
print("Q16 最低折扣要求：")
print(
    analysis["discount_code"]
    .value_counts()
    .sort_index()
)

print("\nQ34 WTP：")
print(
    analysis["wtp_level"]
    .value_counts()
    .sort_index()
)

print("\nQ29 与陌生人拼车意愿：")
print(
    analysis["share_willing"]
    .value_counts()
    .sort_index()
)

print("\nQ24 拼车意愿 Y：")
print(
    analysis["try_willing"]
    .value_counts()
    .sort_index()
)


## 15. 图 1：最低可接受折扣分布


In [ ]:
discount_labels = {
    1: "Fare ratio 0.75",
    2: "Fare ratio 0.65",
    3: "Fare ratio 0.55",
    4: "Never accept"
}

counts = (
    analysis["discount_code"]
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    [discount_labels[int(i)] for i in counts.index],
    counts.values
)

ax.set_ylabel("Number of respondents")
ax.set_xlabel("Minimum acceptable fare ratio")
ax.set_title(
    "Distribution of Minimum Acceptable Discount"
)

plt.tight_layout()
plt.show()


## 16. 图 2：WTP 分布


In [ ]:
wtp_labels = {
    0: "Not willing to pay",
    1: "<5 RMB",
    2: "5-10 RMB",
    3: "10-20 RMB",
    4: ">20 RMB"
}

counts = (
    analysis["wtp_level"]
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    [wtp_labels[int(i)] for i in counts.index],
    counts.values
)

ax.set_ylabel("Number of respondents")
ax.set_xlabel("WTP category")
ax.set_title(
    "Distribution of Stated Willingness-to-Pay"
)

plt.tight_layout()
plt.show()


## 17. 主模型 M1：只使用基础出行特征


In [ ]:
# 主模型采用 Ordinal Logistic Regression
#
# 原因：
# Q24 本来就是 5 级有序拼车意愿，
# 所以不需要强行二分类。

y_ord = (
    analysis["try_willing"]
    .astype(int)
)

X_base = analysis[
    [
        "age_level",
        "income_level",
        "commute_level",
        "vehicle_level"
    ]
].astype(float)

ord_base = OrderedModel(
    y_ord,
    X_base,
    distr="logit"
).fit(
    method="bfgs",
    disp=False
)

print(
    ord_base.summary()
)


## 18. 主模型 M2：加入 Affordability Variables


In [ ]:
# M2：
# 基础特征
# + Q16 minimum acceptable discount
# + Q34 WTP

X_aff = pd.get_dummies(
    analysis[
        [
            "age_level",
            "income_level",
            "commute_level",
            "vehicle_level",
            "discount_code",
            "wtp_level"
        ]
    ],
    columns=[
        "discount_code"
    ],
    drop_first=True,
    dtype=float
)

ord_aff = OrderedModel(
    y_ord,
    X_aff,
    distr="logit"
).fit(
    method="bfgs",
    disp=False
)

print(
    ord_aff.summary()
)


## 19. 主模型 M3：再加入 Sharing Willingness


In [ ]:
# M3：
# M2
# + Q29 willingness to share with strangers

X_full = pd.get_dummies(
    analysis[
        [
            "age_level",
            "income_level",
            "commute_level",
            "vehicle_level",
            "discount_code",
            "wtp_level",
            "share_willing"
        ]
    ],
    columns=[
        "discount_code"
    ],
    drop_first=True,
    dtype=float
)

ord_full = OrderedModel(
    y_ord,
    X_full,
    distr="logit"
).fit(
    method="bfgs",
    disp=False
)

print(
    ord_full.summary()
)


## 20. 比较 M1、M2、M3


In [ ]:
def lr_test(restricted, full):
    """
    Likelihood-Ratio Test

    用来检验：
    加入新的 behavioural variables 后，
    模型拟合是否显著改善。
    """

    lr = 2 * (
        full.llf - restricted.llf
    )

    df_diff = int(
        full.df_model - restricted.df_model
    )

    p_value = chi2.sf(
        lr,
        df_diff
    )

    return lr, df_diff, p_value


# M1 -> M2
lr_aff = lr_test(
    ord_base,
    ord_aff
)

# M2 -> M3
lr_share = lr_test(
    ord_aff,
    ord_full
)


model_compare = pd.DataFrame({
    "Model": [
        "M1: Travel characteristics",
        "M2: + Affordability",
        "M3: + Sharing attitude"
    ],

    "AIC": [
        ord_base.aic,
        ord_aff.aic,
        ord_full.aic
    ],

    "BIC": [
        ord_base.bic,
        ord_aff.bic,
        ord_full.bic
    ],

    "LogLik": [
        ord_base.llf,
        ord_aff.llf,
        ord_full.llf
    ]
})

display(
    model_compare.round(3)
)

print(
    f"M1 -> M2: LR={lr_aff[0]:.3f}, "
    f"df={lr_aff[1]}, "
    f"p={lr_aff[2]:.6g}"
)

print(
    f"M2 -> M3: LR={lr_share[0]:.3f}, "
    f"df={lr_share[1]}, "
    f"p={lr_share[2]:.6g}"
)


## 21. 输出最终 M3 的 Odds Ratio


In [ ]:
# 只提取真正的 X 系数
# 不包含 ordinal model 自己的 threshold 参数

slope_names = list(
    X_full.columns
)

ord_table = pd.DataFrame({
    "Variable": slope_names,

    "Coef": ord_full.params[
        slope_names
    ],

    "StdErr": ord_full.bse[
        slope_names
    ],

    "p_value": ord_full.pvalues[
        slope_names
    ]
})


# Odds Ratio
ord_table["OddsRatio"] = np.exp(
    ord_table["Coef"]
)


# 95% CI 下界
ord_table["CI_low"] = np.exp(
    ord_table["Coef"]
    - 1.96 * ord_table["StdErr"]
)


# 95% CI 上界
ord_table["CI_high"] = np.exp(
    ord_table["Coef"]
    + 1.96 * ord_table["StdErr"]
)


display(
    ord_table.round(4)
)


## 22. 论文主图：Odds Ratio Forest Plot


In [ ]:
# 把模型里的变量名改成论文更容易读的名字

pretty_names = {
    "age_level": "Age",
    "income_level": "Income",
    "commute_level": "Commute distance",
    "vehicle_level": "Vehicle ownership",

    "discount_code_2.0":
        "Discount requirement: fare ratio 0.65",

    "discount_code_3.0":
        "Discount requirement: fare ratio 0.55",

    "discount_code_4.0":
        "Never accept regardless of discount",

    "wtp_level":
        "Willingness-to-pay category",

    "share_willing":
        "Willingness to share with strangers"
}


forest = ord_table.copy()

forest["Label"] = (
    forest["Variable"]
    .map(pretty_names)
    .fillna(forest["Variable"])
)

# 反转顺序，让第一项显示在图的上方
forest = (
    forest
    .iloc[::-1]
    .reset_index(drop=True)
)


fig, ax = plt.subplots(
    figsize=(9, 6)
)


y_pos = np.arange(
    len(forest)
)


# 计算误差条长度
lower_error = (
    forest["OddsRatio"]
    - forest["CI_low"]
)

upper_error = (
    forest["CI_high"]
    - forest["OddsRatio"]
)


# 画 OR 和 95% CI
ax.errorbar(
    forest["OddsRatio"],
    y_pos,

    xerr=np.vstack([
        lower_error,
        upper_error
    ]),

    fmt="o",
    capsize=3
)


# OR = 1 表示没有关联
ax.axvline(
    1,
    linestyle="--"
)


ax.set_yticks(
    y_pos
)

ax.set_yticklabels(
    forest["Label"]
)


# Odds Ratio 一般用 log scale 更容易阅读
ax.set_xscale(
    "log"
)

ax.set_xlabel(
    "Odds Ratio (95% CI)"
)

ax.set_title(
    "Factors Associated with Ride-Pooling Willingness"
)


plt.tight_layout()


# 保存为期刊可用的高分辨率 PNG
plt.savefig(
    "fig_main_forest_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 23. 保存主模型结果


In [ ]:
# 保存主模型比较结果

model_compare.to_csv(
    "ordinal_model_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)


# 保存最终 M3 Odds Ratio 结果

ord_table.to_csv(
    "ordinal_logit_results.csv",
    index=False,
    encoding="utf-8-sig"
)


print("已保存：")
print("ordinal_model_comparison.csv")
print("ordinal_logit_results.csv")
print("fig_main_forest_plot.png")
